In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
columns = [
    'checking_account',     # Status of existing checking account
    'duration',             # Duration in months
    'credit_history',       # Credit history
    'purpose',              # Purpose of loan
    'credit_amount',        # Credit amount
    'savings_account',      # Savings account/bonds
    'employment_since',     # Present employment since
    'installment_rate',     # Installment rate (% of disposable income)
    'personal_status',      # Personal status and sex
    'other_debtors',        # Other debtors/guarantors
    'residence_since',      # Present residence since
    'property',             # Property
    'age',                  # Age in years
    'other_installments',   # Other installment plans
    'housing',              # Housing
    'existing_credits',     # Number of existing credits at this bank
    'job',                  # Job
    'liable_people',        # Number of people liable to provide maintenance
    'telephone',            # Telephone
    'foreign_worker',       # Foreign worker
    'target'                # 1 = Good, 2 = Bad
]

In [3]:
df = pd.read_csv('../data/original/german_credit_data.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns ✓")

Dataset loaded: 1000 rows, 21 columns ✓


In [4]:
# Shape
print("=== SHAPE ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Data types
print("\n=== DATA TYPES ===")
print(df.dtypes)

# First look
print("\n=== FIRST 5 ROWS ===")
df.head()

=== SHAPE ===
Rows: 1000, Columns: 21

=== DATA TYPES ===
checking_status             str
duration                  int64
credit_history              str
purpose                     str
credit_amount             int64
savings_status              str
employment                  str
installment_commitment    int64
personal_status             str
other_parties               str
residence_since           int64
property_magnitude          str
age                       int64
other_payment_plans         str
housing                     str
existing_credits          int64
job                         str
num_dependents            int64
own_telephone               str
foreign_worker              str
class                       str
dtype: object

=== FIRST 5 ROWS ===


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,no known property,53,none,for free,2,skilled,2,none,yes,bad


check for comma-seperated or space-seperated csv file


In [5]:
# First, peek at the raw file to confirm the separator
with open('../data/original/german_credit_data.csv', 'r') as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i == 3:
            break

'checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class\n'
'<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes,good\n'
'0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes,bad\n'
'no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes,good\n'


In [6]:
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print(f"\n=== DUPLICATES ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

=== MISSING VALUES ===
checking_status           0
duration                  0
credit_history            0
purpose                   0
credit_amount             0
savings_status            0
employment                0
installment_commitment    0
personal_status           0
other_parties             0
residence_since           0
property_magnitude        0
age                       0
other_payment_plans       0
housing                   0
existing_credits          0
job                       0
num_dependents            0
own_telephone             0
foreign_worker            0
class                     0
dtype: int64

=== DUPLICATES ===
Duplicate rows: 0


In [7]:
print("=== TARGET DISTRIBUTION ===")
print(df['class'].value_counts())
print(f"\nClass balance: {df['class'].value_counts(normalize=True).round(3) * 100}")

=== TARGET DISTRIBUTION ===
class
good    700
bad     300
Name: count, dtype: int64

Class balance: class
good    70.0
bad     30.0
Name: proportion, dtype: float64


In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols 

['duration',
 'credit_amount',
 'installment_commitment',
 'residence_since',
 'age',
 'existing_credits',
 'num_dependents']

Truly continuous => duration, credit_amount, age

Ordinal numeric => installment_commitment, residence_since, existing_credits, num_dependants



In [9]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = categorical_cols[:-1]  # Exclude target
categorical_cols

['checking_status',
 'credit_history',
 'purpose',
 'savings_status',
 'employment',
 'personal_status',
 'other_parties',
 'property_magnitude',
 'other_payment_plans',
 'housing',
 'job',
 'own_telephone',
 'foreign_worker']

## Univariate EDA

Univariate means one variable at a time. like  asking: "What does this variable look like on its own?"
For continuous variables we look at:

- Distribution — is it normal, skewed, bimodal?
- Outliers — are there extreme values?
- Central tendency — where does most data cluster?

In [10]:
df[numeric_cols].describe().round(2)

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,20.90,3271.26,2.97,2.84,35.55,1.41,1.16
std,12.06,2822.74,1.12,1.10,11.38,0.58,0.36
min,4.00,250.00,1.00,1.00,19.00,1.00,1.00
25%,12.00,1365.50,2.00,2.00,27.00,1.00,1.00
50%,18.00,2319.50,3.00,3.00,33.00,1.00,1.00
75%,24.00,3972.25,4.00,4.00,42.00,2.00,1.00
max,72.00,18424.00,4.00,4.00,75.00,4.00,2.00


In [11]:
continuous_cols = ['duration', 'credit_amount', 'age']

for col in continuous_cols:
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=(f'{col} — Distribution', 
                                        f'{col} — Boxplot'))
    
    # Histogram
    fig.add_trace(
        go.Histogram(x=df[col], nbinsx=30,
                     marker_color='steelblue', opacity=0.75,
                     name='Distribution'),
        row=1, col=1
    )
    
    # Boxplot
    fig.add_trace(
        go.Box(y=df[col], marker_color='steelblue',
               name='Boxplot', boxmean=True),
        row=1, col=2
    )
    
    fig.update_layout(
        title=f'Univariate Analysis — {col}',
        height=400, showlegend=False,
        template='plotly_dark'
    )
    fig.show()

In [14]:
# Check the actual outlier count for credit_amount
Q1 = df['credit_amount'].quantile(0.25)
Q3 = df['credit_amount'].quantile(0.75)
IQR = Q3 - Q1

outliers = df[(df['credit_amount'] < Q1 - 1.5*IQR) | 
              (df['credit_amount'] > Q3 + 1.5*IQR)]

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Outlier count: {len(outliers)}")
print(f"Outlier % : {len(outliers)/len(df)*100:.1f}%")

Q1: 1365.5, Q3: 3972.25, IQR: 2606.75
Outlier count: 72
Outlier % : 7.2%


In [13]:
ordinal_cols = ['installment_commitment', 'residence_since', 
                'existing_credits', 'num_dependents']

for col in ordinal_cols:
    counts = df[col].value_counts().sort_index()
    
    fig = px.bar(x=counts.index, y=counts.values,
                 labels={'x': col, 'y': 'Count'},
                 title=f'Univariate Analysis — {col}',
                 color=counts.values,
                 color_continuous_scale='Blues',
                 template='plotly_dark')
    
    fig.update_layout(height=350, showlegend=False)
    fig.show()

## Univariate EDA — Key Findings

| Variable | Finding | Action |
|---|---|---|
| credit_amount | Right skewed, 72 outliers (7.2%) | Log transform on Day 2 |
| duration | Skewed, clusters at 12–24 months | Check long loans vs default rate |
| age | Slight right skew, outliers at high end | Monitor in bivariate |
| installment_rate | Most borrowers at rate=4 (highest burden) | Strong risk signal — test in hypothesis testing |
| existing_credits | 333 people have 2 existing loans | Engineer debt burden feature on Day 2 |